**Reviewer note:** This notebook is a preserved development experiment. Read `docs/EXPERIMENTS.md` and `docs/METHODOLOGY.md` before execution. Raw/restricted inputs are not included. Historical references to 'test' do not establish prospective independence or validated online causality.

# 0. Controlli post-training FEM v2 con CV temporale annidata

Questo notebook non ripete lo screening completo del notebook 40. Esegue i
controlli causali mancanti esclusivamente su EL e su 24.000 finestre FEM:

1. corregge l'early stopping: l'outer fold non viene mai passato a Keras;
2. aggiunge F1A, cioe' FEM v1 con la stessa testa reinizializzata di F2A;
3. confronta F1A e F2A con inizializzazioni e budget appaiati;
4. rende condizionale il bilanciamento per elevazione;
5. prova un correttore residuale cross-fitted della Ridge con `lambda=0` ammesso;
6. verifica la disponibilita' degli operatori fisici per il test su telemetria reale.

Il test reale da 135 casi non e' accessibile da questo notebook.

In [ ]:
# 1. Configurazione preregistrata e import
from pathlib import Path
import gc
import hashlib
import json
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_SEEDS = [17, 29, 43]
FEM_WINDOW_CURVE = [1_000, 3_000, 8_000, 24_000]
COMMON_ELEVATIONS = [30.0, 60.0, 85.0]
SEQ_LEN = 60
N_PCA = 16
BATCH_SIZE = 128
PRETRAIN_EPOCHS = 40
HEAD_EPOCHS = 20
FINETUNE_EPOCHS = 60
SSL_EPOCHS = 40
TEMPORAL_CV_FOLDS = 3
MAX_SSL_WINDOWS = 12_000
SPEC_ARCSEC = 5.0
MATERIAL_ABSOLUTE_ARCSEC = 0.30
MATERIAL_RELATIVE = 0.03
MAX_SINGLE_FOLD_REGRESSION = 0.25
LAMBDA_GRID = [0.0, 0.25, 0.50, 0.75, 1.0]

RUN_BALANCED_IF_TEST_A_PASSES = True
RUN_RESIDUAL_CORRECTOR = True
RUN_PHYSICAL_LAMBDA_IF_TEST_A_PASSES = True
RUN_LOCKED_TEST = False

tf.keras.utils.set_random_seed(RANDOM_SEEDS[0])
sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 2. Percorsi e caricamento dei 668 cross-scan reali
def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/metrologyml")]
    for candidate in candidates:
        if (candidate / "src" / "aims_pipeline.py").exists():
            return candidate.resolve()
    raise FileNotFoundError("Cartella ML_metrologia non trovata")


ROOT = find_project_root()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from fem_v2_labels import attach_fem_v2_labels
from aims_pipeline import ThermalFeatureBuilder, prepare_samples, receiver_one_hot
from aims_training import el_model, xel_model

REAL_PATH = ROOT / "data" / "raw" / "real" / "EffectiveDataset_Tseries_with_azimuth.csv"
TARGET_AUDIT_PATH = ROOT / "data" / "raw" / "control" / "control_crosscheck_effective_target_projection.csv"
POINTING_AUDIT_PATH = ROOT / "data" / "raw" / "control" / "control_crosscheck_pointing_audit.csv"
FEM_SOURCE_PATH = ROOT / "data" / "raw" / "fem" / "Dataset (1).xlsx"
FEM_V2_LABELS_PATH = ROOT / "data" / "processed" / "fem_v2_common_only_labels_30_60_85.csv"
FEM_V2_MANIFEST_PATH = ROOT / "data" / "processed" / "fem_v2_common_only_labels_30_60_85.manifest.json"
OUTPUT_DIR = ROOT / "outputs" / "experiments" / "fem_v2_common_only"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

samples = prepare_samples(REAL_PATH, TARGET_AUDIT_PATH, POINTING_AUDIT_PATH)
X_real = samples.sequences.astype("float32")
meta = samples.metadata.reset_index(drop=True).copy()
meta["Timestamp"] = pd.to_datetime(meta["Timestamp"])
y_real = {
    "XEL": samples.target_xel.astype("float32"),
    "EL": samples.target_el.astype("float32"),
}
train_idx = np.flatnonzero(meta["Split"].eq("train"))
val_idx = np.flatnonzero(meta["Split"].eq("validation"))
test_idx = np.flatnonzero(meta["Split"].eq("test"))
if (len(meta), len(train_idx), len(val_idx), len(test_idx)) != (668, 400, 133, 135):
    raise ValueError("Split reale inatteso: il test bloccato non puo' essere ridefinito")

real_data_summary_df = pd.DataFrame([{
    "samples": len(meta), "train": len(train_idx), "validation": len(val_idx),
    "locked_test": len(test_idx), "sequence_shape": str(tuple(X_real.shape[1:])),
    "validation_start": meta.loc[val_idx, "Timestamp"].min(),
    "test_start": meta.loc[test_idx, "Timestamp"].min(),
    "locked_test_used": False,
}])
display(real_data_summary_df)

In [ ]:
# 3. Join FEM v2 verificato prima di filtri, deduplicazione e resampling
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


manifest = json.loads(FEM_V2_MANIFEST_PATH.read_text(encoding="utf-8"))
source_hash = sha256_file(FEM_SOURCE_PATH)
labels_hash = sha256_file(FEM_V2_LABELS_PATH)
if source_hash.lower() != manifest["source_workbook_sha256"].lower():
    raise ValueError("Hash del workbook FEM diverso dal manifest")
if labels_hash.lower() != manifest["labels_sha256"].lower():
    raise ValueError("Hash delle label FEM v2 diverso dal manifest")

print("Caricamento workbook FEM da 356.400 righe...")
fem_raw = pd.read_excel(FEM_SOURCE_PATH, engine="openpyxl")
source_rows = len(fem_raw)
rows_in_duplicate_groups_raw = int(
    fem_raw.duplicated(["Time", "Elevation"], keep=False).sum()
)
duplicate_excess_raw = int(
    fem_raw.duplicated(["Time", "Elevation"], keep="first").sum()
)
fem_common = attach_fem_v2_labels(fem_raw, FEM_V2_LABELS_PATH)
timestamp_alignment_max_abs_ms = fem_common.attrs.get(
    "timestamp_alignment_max_abs_ms", np.nan
)
fem_common["Time"] = pd.to_datetime(fem_common["Time"], format="mixed", errors="raise")
fem_common["Elevation"] = pd.to_numeric(fem_common["Elevation"], errors="raise")
if sorted(fem_common["Elevation"].unique().astype(float)) != COMMON_ELEVATIONS:
    raise ValueError("Le elevazioni comuni non sono 30, 60 e 85 gradi")

sensor_cols = sorted(
    [column for column in fem_common.columns if re.match(r"^Sensors_\s*\d+$", str(column))],
    key=lambda column: int(re.search(r"(\d+)$", str(column)).group(1)),
)
if len(sensor_cols) != 295:
    raise ValueError(f"Attesi 295 sensori FEM, trovati {len(sensor_cols)}")
fem_common = fem_common.rename(columns={
    column: "Sensors_{:03d}".format(
        int(re.search(r"(\d+)$", str(column)).group(1))
    )
    for column in sensor_cols
})
fem_sensor_cols = sorted(
    [column for column in fem_common.columns if re.match(r"^Sensors_\d{3}$", column)],
    key=lambda column: int(column.split("_")[1]),
)
required_numeric = fem_sensor_cols + [
    "FEMv1_XEL_arcsec", "FEMv1_EL_arcsec",
    "FEMv2_XEL_arcsec", "FEMv2_EL_arcsec",
]
for column in required_numeric:
    fem_common[column] = pd.to_numeric(fem_common[column], errors="coerce")
fem_common = fem_common.dropna(subset=["Time", "Elevation", *required_numeric]).copy()

fem_join_audit_df = pd.DataFrame([{
    "source_rows": source_rows, "matched_common_rows": len(fem_common),
    "sensor_count": len(fem_sensor_cols),
    "rows_in_duplicate_groups_raw": rows_in_duplicate_groups_raw,
    "duplicate_excess_rows_raw": duplicate_excess_raw,
    "timestamp_alignment_max_abs_ms": timestamp_alignment_max_abs_ms,
    "source_sha256": source_hash, "labels_sha256": labels_hash,
    "elevations": ",".join(map(lambda value: str(int(value)), COMMON_ELEVATIONS)),
    "v2_XEL_cosine_applied": False,
}])
display(fem_join_audit_df)
display(
    fem_common.groupby("Elevation", as_index=False)
    .agg(
        rows=("source_row_id", "size"),
        v1_XEL_std=("FEMv1_XEL_arcsec", "std"),
        v2_XEL_std=("FEMv2_XEL_arcsec", "std"),
        v1_EL_std=("FEMv1_EL_arcsec", "std"),
        v2_EL_std=("FEMv2_EL_arcsec", "std"),
    )
)
del fem_raw
gc.collect()

In [ ]:
# 4. Rappresentazione termica definita esclusivamente sul training reale
class RealDomainThermalProjector:
    def __init__(self, n_components=N_PCA):
        self.mean_scaler = StandardScaler()
        self.pattern_scaler = StandardScaler()
        self.pattern_pca = PCA(n_components=n_components, random_state=RANDOM_SEEDS[0])

    def fit(self, sequences):
        frames = np.asarray(sequences, dtype="float32").reshape(-1, sequences.shape[-1])
        means = frames.mean(axis=1, keepdims=True)
        centered = frames - means
        self.mean_scaler.fit(means)
        self.pattern_pca.fit(self.pattern_scaler.fit_transform(centered))
        return self

    def transform_frames(self, frames):
        frames = np.asarray(frames, dtype="float32")
        means = frames.mean(axis=1, keepdims=True)
        centered = frames - means
        return np.concatenate([
            self.mean_scaler.transform(means),
            self.pattern_pca.transform(self.pattern_scaler.transform(centered)),
        ], axis=1).astype("float32")

    def transform(self, sequences):
        shape = sequences.shape
        values = self.transform_frames(sequences.reshape(-1, shape[-1]))
        return values.reshape(shape[0], shape[1], -1)


RECEIVER_LEVELS = ["K", "C_high", "C_low", "Q", "unknown"]


def context_features(elevation, receiver_band=None):
    elev = np.asarray(elevation, dtype="float32").reshape(-1)
    if receiver_band is None:
        receiver_band = np.repeat("unknown", len(elev))
    category = pd.Categorical(receiver_band, categories=RECEIVER_LEVELS)
    one_hot = pd.get_dummies(category, dtype=float).to_numpy(dtype="float32")
    return np.column_stack([
        elev / 90.0, np.sin(np.deg2rad(elev)), np.cos(np.deg2rad(elev)), one_hot,
    ]).astype("float32")


projector = RealDomainThermalProjector().fit(X_real[train_idx])
Z_real = projector.transform(X_real)
C_real = context_features(meta["Elevation"], meta["ReceiverBand"].fillna("unknown"))
representation_summary_df = pd.DataFrame([{
    "fit_domain": "real_train_only", "train_sequences": len(train_idx),
    "PCA_components": N_PCA, "sequence_features": Z_real.shape[-1],
    "context_features": C_real.shape[-1],
    "pattern_variance_explained_percent": 100 * projector.pattern_pca.explained_variance_ratio_.sum(),
    "locked_test_used_for_fit": False,
}])
display(representation_summary_df)

In [ ]:
# 5. Banca canonica di finestre FEM e split per gruppi giorno-elevazione
LABEL_COLUMNS = [
    "FEMv1_XEL_arcsec", "FEMv1_EL_arcsec",
    "FEMv2_XEL_arcsec", "FEMv2_EL_arcsec",
]


def build_fem_group_bank(frame, sensors):
    groups, candidates = {}, []
    rows_in_duplicate_groups = 0
    duplicate_excess_rows = 0
    work = frame.copy()
    work["day"] = work["Time"].dt.floor("D")
    for (elevation, day), group in work.groupby(["Elevation", "day"], sort=True):
        group = group.sort_values(["Time", "source_row_id"])
        rows_in_duplicate_groups += int(group["Time"].duplicated(keep=False).sum())
        duplicate_excess_rows += int(group["Time"].duplicated(keep="first").sum())
        values = group.set_index("Time")
        averaged = values[sensors + LABEL_COLUMNS].resample("2min").mean()
        source_ids = values[["source_row_id"]].resample("2min").max()
        sampled = pd.concat([averaged, source_ids], axis=1).dropna()
        if len(sampled) < SEQ_LEN:
            continue
        group_id = f"{pd.Timestamp(day).date()}__{int(round(elevation))}"
        groups[group_id] = {
            "sensors": sampled[sensors].to_numpy(dtype="float32"),
            "labels": sampled[LABEL_COLUMNS].to_numpy(dtype="float32"),
            "times": sampled.index.to_numpy(),
            "source_ids": sampled["source_row_id"].round().astype("int64").to_numpy(),
            "elevation": float(elevation), "day": pd.Timestamp(day),
        }
        for endpoint in range(SEQ_LEN - 1, len(sampled)):
            candidates.append({
                "candidate_id": len(candidates), "group_id": group_id,
                "day": pd.Timestamp(day), "elevation": float(elevation),
                "endpoint": endpoint, "end_timestamp": sampled.index[endpoint],
                "end_source_row_id": int(sampled["source_row_id"].iloc[endpoint]),
            })
    return (
        groups, pd.DataFrame(candidates),
        rows_in_duplicate_groups, duplicate_excess_rows,
    )


def assign_group_splits(candidates):
    group_table = candidates[["group_id", "day", "elevation"]].drop_duplicates()
    assignments = []
    for _, subset in group_table.groupby("elevation", sort=True):
        subset = subset.sort_values(["day", "group_id"]).reset_index(drop=True)
        cut = max(1, int(np.floor(0.90 * len(subset))))
        subset["fem_split"] = np.where(subset.index < cut, "train", "validation")
        assignments.append(subset)
    assignment = pd.concat(assignments, ignore_index=True)
    result = candidates.merge(
        assignment[["group_id", "fem_split"]], on="group_id", validate="many_to_one"
    )
    overlap = set(result.query("fem_split == 'train'")["group_id"]) & set(
        result.query("fem_split == 'validation'")["group_id"]
    )
    if overlap:
        raise RuntimeError("Leakage nei gruppi FEM")
    return result


(
    fem_groups, fem_candidates,
    rows_in_duplicate_groups_common, duplicate_excess_common,
) = build_fem_group_bank(fem_common, fem_sensor_cols)
fem_candidates = assign_group_splits(fem_candidates)
fixed_rng = np.random.default_rng(20260921)
train_candidate_ids = fem_candidates.query("fem_split == 'train'")[
    "candidate_id"
].to_numpy(dtype=int)
fixed_rng.shuffle(train_candidate_ids)
validation_candidate_ids = fem_candidates.query("fem_split == 'validation'")[
    "candidate_id"
].to_numpy(dtype=int)
if len(train_candidate_ids) < max(FEM_WINDOW_CURVE):
    raise ValueError("Finestre FEM insufficienti per la curva preregistrata")


def materialize_fem(candidate_ids, fitted_projector):
    selected = fem_candidates.set_index("candidate_id").loc[np.asarray(candidate_ids)]
    selected_groups = set(selected["group_id"])
    transformed = {
        group_id: fitted_projector.transform_frames(group["sensors"])
        for group_id, group in fem_groups.items() if group_id in selected_groups
    }
    x_out, c_out, y1_out, y2_out, key_rows = [], [], [], [], []
    for candidate_id, row in selected.iterrows():
        group = fem_groups[row["group_id"]]
        endpoint = int(row["endpoint"])
        x_out.append(transformed[row["group_id"]][endpoint - SEQ_LEN + 1:endpoint + 1])
        c_out.append(context_features([row["elevation"]])[0])
        labels = group["labels"][endpoint]
        y1_out.append(labels[:2])
        y2_out.append(labels[2:])
        key_rows.append({
            "candidate_id": int(candidate_id), "group_id": row["group_id"],
            "day": row["day"], "elevation": row["elevation"],
            "end_timestamp": row["end_timestamp"],
            "end_source_row_id": row["end_source_row_id"],
        })
    return {
        "Z": np.asarray(x_out, dtype="float32"),
        "C": np.asarray(c_out, dtype="float32"),
        "Y_v1": np.asarray(y1_out, dtype="float32"),
        "Y_v2": np.asarray(y2_out, dtype="float32"),
        "keys": pd.DataFrame(key_rows),
    }


fem_train_bank = materialize_fem(train_candidate_ids[:max(FEM_WINDOW_CURVE)], projector)
fem_validation_bank = materialize_fem(validation_candidate_ids, projector)
if not fem_train_bank["keys"]["candidate_id"].is_unique:
    raise RuntimeError("Chiavi delle finestre FEM non uniche")
fem_window_summary_df = pd.DataFrame([{
    "common_rows": len(fem_common), "groups": len(fem_groups),
    "train_groups": fem_candidates.query("fem_split == 'train'")["group_id"].nunique(),
    "validation_groups": fem_candidates.query("fem_split == 'validation'")["group_id"].nunique(),
    "candidate_windows": len(fem_candidates),
    "fixed_train_windows": len(fem_train_bank["Z"]),
    "validation_windows": len(fem_validation_bank["Z"]),
    "rows_in_duplicate_groups_common_elevations": rows_in_duplicate_groups_common,
    "duplicate_excess_rows_common_elevations": duplicate_excess_common,
    "sequence_shape": str(tuple(fem_train_bank["Z"].shape[1:])),
}])
display(fem_window_summary_df)
display(fem_train_bank["keys"].head())
del fem_common
gc.collect()

In [ ]:
# 6. Telemetria reale non etichettata precedente alla validation
def normalized_sensor_name(name):
    match = re.match(r"^Sensors_\s*(\d+)$", str(name).strip())
    return f"Sensors_{int(match.group(1)):03d}" if match else str(name).strip()


def valid_endpoints(times, seq_len=SEQ_LEN, max_gap_seconds=180):
    times = pd.to_datetime(times).to_numpy(dtype="datetime64[ns]")
    delta = np.diff(times).astype("timedelta64[s]").astype(float)
    bad = (delta > max_gap_seconds) | (delta <= 0)
    prefix = np.r_[0, np.cumsum(bad.astype(int))]
    return np.asarray([
        end for end in range(seq_len - 1, len(times))
        if prefix[end] - prefix[end - seq_len + 1] == 0
    ], dtype=int)


def load_ssl_sequences(fitted_projector, cutoff, seed, max_windows=MAX_SSL_WINDOWS):
    raw = pd.read_csv(REAL_PATH, low_memory=False).rename(columns=normalized_sensor_name)
    raw["Timestamp"] = pd.to_datetime(raw["Timestamp"], errors="coerce")
    sensors = sorted(
        [column for column in raw.columns if re.match(r"^Sensors_\d{3}$", column)],
        key=lambda column: int(column.split("_")[1]),
    )
    before = raw.loc[raw["Timestamp"].lt(cutoff), ["Timestamp", *sensors]].dropna()
    before["GridTime"] = before["Timestamp"].dt.floor("2min")
    grid = before.groupby("GridTime", sort=True)[sensors].mean().ffill(limit=3).dropna()
    endpoints = valid_endpoints(grid.index)
    rng = np.random.default_rng(seed)
    if len(endpoints) > max_windows:
        endpoints = np.sort(rng.choice(endpoints, max_windows, replace=False))
    frames = fitted_projector.transform_frames(grid.to_numpy(dtype="float32"))
    sequences = np.stack([
        frames[end - SEQ_LEN + 1:end + 1] for end in endpoints
    ]).astype("float32")
    return sequences, {
        "telemetry_rows_before_cutoff": len(before), "ssl_windows": len(sequences),
        "cutoff": cutoff, "first_time": grid.index.min(), "last_time": grid.index.max(),
        "future_rows_used": False,
    }


Z_ssl, ssl_summary = load_ssl_sequences(
    projector, meta.loc[val_idx, "Timestamp"].min(), RANDOM_SEEDS[0]
)
ssl_data_summary_df = pd.DataFrame([ssl_summary])
display(ssl_data_summary_df)

In [ ]:
# 7. Architettura comune, inizializzazioni appaiate e metriche
AXIS_INDEX = {"XEL": 0, "EL": 1}


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    residual = y_true - y_pred
    absolute = np.abs(residual)
    return {
        "count": len(y_true), "MAE": float(absolute.mean()),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)), "bias": float(residual.mean()),
        "within_5_percent": float(100 * np.mean(absolute <= SPEC_ARCSEC)),
        "p90_abs_error": float(np.quantile(absolute, 0.90)),
    }


def make_regressor(input_dim, context_dim):
    sequence = keras.Input((SEQ_LEN, input_dim), name="sequence")
    context = keras.Input((context_dim,), name="context")
    encoded = layers.GRU(32, dropout=0.10, name="thermal_gru")(sequence)
    encoded = layers.Dense(16, activation="gelu", name="thermal_projection")(encoded)
    ctx = layers.Dense(12, activation="gelu", name="context_projection")(context)
    fused = layers.Concatenate(name="fusion")([encoded, ctx])
    hidden = layers.Dense(20, activation="gelu", name="supervised_hidden")(fused)
    hidden = layers.Dropout(0.20, name="supervised_dropout")(hidden)
    output = layers.Dense(1, name="supervised_output")(hidden)
    return keras.Model([sequence, context], output)


def layer_state(model):
    return {layer.name: layer.get_weights() for layer in model.layers if layer.get_weights()}


def restore_state(model, state, excluded=()):
    for layer in model.layers:
        if layer.name in state and layer.name not in excluded and layer.get_weights():
            layer.set_weights(state[layer.name])
    return model


def callbacks(patience=8):
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", patience=max(3, patience // 2), factor=0.5,
            min_lr=1e-6, verbose=0,
        ),
    ]


def run_ssl_pretraining(Z, seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    sequence = keras.Input((SEQ_LEN, Z.shape[-1]))
    encoded = layers.GRU(32, dropout=0.10, name="thermal_gru")(sequence)
    encoded = layers.Dense(16, activation="gelu", name="thermal_projection")(encoded)
    repeated = layers.RepeatVector(SEQ_LEN)(encoded)
    decoded = layers.GRU(32, return_sequences=True)(repeated)
    output = layers.TimeDistributed(layers.Dense(Z.shape[-1]))(decoded)
    model = keras.Model(sequence, output)
    rng = np.random.default_rng(seed)
    corrupted = Z.copy()
    mask = rng.random(corrupted.shape[:2]) < 0.15
    corrupted[mask] = 0.0
    corrupted += rng.normal(0, 0.02, corrupted.shape).astype("float32")
    cut = max(1, int(0.90 * len(Z)))
    model.compile(keras.optimizers.Adam(3e-4, clipnorm=1.0), loss="mse")
    model.fit(
        corrupted[:cut], Z[:cut], validation_data=(corrupted[cut:], Z[cut:]),
        epochs=SSL_EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks(), verbose=0,
    )
    return {
        name: model.get_layer(name).get_weights()
        for name in ["thermal_gru", "thermal_projection"]
    }


def run_fem_pretraining(bank_train, bank_val, axis, label_version, sample_count, seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    axis_index = AXIS_INDEX[axis]
    y_key = "Y_v1" if label_version == "v1" else "Y_v2"
    y_train = bank_train[y_key][:sample_count, axis_index]
    y_val = bank_val[y_key][:, axis_index]
    scaler = StandardScaler().fit(y_train.reshape(-1, 1))
    model = make_regressor(bank_train["Z"].shape[-1], bank_train["C"].shape[-1])
    model.compile(keras.optimizers.Adam(3e-4, clipnorm=1.0), loss=keras.losses.Huber())
    model.fit(
        [bank_train["Z"][:sample_count], bank_train["C"][:sample_count]],
        scaler.transform(y_train.reshape(-1, 1)),
        validation_data=(
            [bank_val["Z"], bank_val["C"]], scaler.transform(y_val.reshape(-1, 1))
        ),
        epochs=PRETRAIN_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks(), verbose=0,
    )
    return layer_state(model)


def fit_real_regressor(
    Z_train, C_train, y_train, Z_eval, C_eval, y_eval, seed,
    pretrained_state=None, ssl_state=None, reset_supervised_head=False,
):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    scaler = StandardScaler().fit(np.asarray(y_train).reshape(-1, 1))
    model = make_regressor(Z_train.shape[-1], C_train.shape[-1])
    if pretrained_state is not None:
        excluded = (
            "context_projection", "supervised_hidden", "supervised_output"
        ) if reset_supervised_head else ()
        restore_state(model, pretrained_state, excluded=excluded)
    if ssl_state is not None:
        restore_state(model, ssl_state)
    if pretrained_state is not None or ssl_state is not None:
        model.get_layer("thermal_gru").trainable = False
        model.get_layer("thermal_projection").trainable = False
        model.compile(keras.optimizers.Adam(5e-4, clipnorm=1.0), loss=keras.losses.Huber())
        model.fit(
            [Z_train, C_train], scaler.transform(np.asarray(y_train).reshape(-1, 1)),
            validation_data=(
                [Z_eval, C_eval], scaler.transform(np.asarray(y_eval).reshape(-1, 1))
            ), epochs=HEAD_EPOCHS, batch_size=64, callbacks=callbacks(6), verbose=0,
        )
        model.get_layer("thermal_gru").trainable = True
        model.get_layer("thermal_projection").trainable = True
    model.compile(keras.optimizers.Adam(1e-4, clipnorm=1.0), loss=keras.losses.Huber())
    model.fit(
        [Z_train, C_train], scaler.transform(np.asarray(y_train).reshape(-1, 1)),
        validation_data=(
            [Z_eval, C_eval], scaler.transform(np.asarray(y_eval).reshape(-1, 1))
        ), epochs=FINETUNE_EPOCHS, batch_size=64, callbacks=callbacks(10), verbose=0,
    )
    prediction = scaler.inverse_transform(
        model.predict([Z_eval, C_eval], verbose=0)
    ).reshape(-1)
    return prediction, layer_state(model), scaler


def production_prediction(axis, fit_idx, eval_idx):
    builder = ThermalFeatureBuilder().fit(X_real[fit_idx])
    fit_meta = meta.iloc[fit_idx].reset_index(drop=True)
    eval_meta = meta.iloc[eval_idx].reset_index(drop=True)
    fit_features = builder.transform(
        X_real[fit_idx], samples.elevation[fit_idx], fit_meta
    )
    fit_features = pd.concat([
        fit_features, receiver_one_hot(fit_meta)
    ], axis=1)
    eval_features = builder.transform(
        X_real[eval_idx], samples.elevation[eval_idx], eval_meta
    )
    eval_features = pd.concat([
        eval_features, receiver_one_hot(eval_meta)
    ], axis=1)
    model = (xel_model() if axis == "XEL" else el_model()).fit(
        fit_features, y_real[axis][fit_idx]
    )
    return model.predict(eval_features)

In [ ]:
# 8. Bootstrap appaiato per giornate e raccolta delle predizioni
def day_block_bootstrap(y_true, candidate, reference, timestamps, n_boot=5000, seed=31):
    table = pd.DataFrame({
        "y": np.asarray(y_true), "candidate": np.asarray(candidate),
        "reference": np.asarray(reference), "time": pd.to_datetime(timestamps),
    })
    table["day"] = table["time"].dt.floor("D")
    blocks = [np.asarray(index, dtype=int) for index in table.groupby("day").groups.values()]
    rng = np.random.default_rng(seed)
    deltas = []
    for _ in range(n_boot):
        chosen = rng.integers(0, len(blocks), size=len(blocks))
        idx = np.concatenate([blocks[value] for value in chosen])
        candidate_mae = np.abs(table.loc[idx, "y"] - table.loc[idx, "candidate"]).mean()
        reference_mae = np.abs(table.loc[idx, "y"] - table.loc[idx, "reference"]).mean()
        deltas.append(candidate_mae - reference_mae)
    values = np.asarray(deltas)
    return {
        "delta_MAE": float(values.mean()),
        "ci95_low": float(np.quantile(values, 0.025)),
        "ci95_high": float(np.quantile(values, 0.975)),
        "probability_better_percent": float(100 * np.mean(values < 0)),
    }


def append_predictions(rows, stage, axis, variant, seed, fold, fem_windows, indices, prediction):
    for row_index, value in zip(indices, prediction):
        rows.append({
            "stage": stage, "axis": axis, "variant": variant, "seed": seed,
            "fold": fold, "fem_windows": fem_windows, "row_index": int(row_index),
            "Timestamp": meta.loc[row_index, "Timestamp"],
            "Elevation": float(meta.loc[row_index, "Elevation"]),
            "target": float(y_real[axis][row_index]), "prediction": float(value),
        })


def metrics_from_prediction_rows(frame):
    rows = []
    grouping = ["stage", "axis", "variant", "seed", "fold", "fem_windows"]
    for keys, subset in frame.groupby(grouping, dropna=False, sort=False):
        rows.append(dict(zip(grouping, keys)) | regression_metrics(
            subset["target"], subset["prediction"]
        ))
    return pd.DataFrame(rows)

In [ ]:
# 9. Training annidato: scelta epoche su inner validation e refit su outer train
def temporal_fold_bounds(indices, n_folds=TEMPORAL_CV_FOLDS):
    indices = np.asarray(indices)
    initial = max(180, len(indices) // 2)
    remaining = len(indices) - initial
    step = max(1, remaining // n_folds)
    folds = []
    for fold in range(n_folds):
        train_end = initial + fold * step
        eval_end = len(indices) if fold == n_folds - 1 else min(len(indices), train_end + step)
        if eval_end > train_end:
            folds.append((indices[:train_end], indices[train_end:eval_end]))
    return folds


def inner_time_split(outer_train, validation_fraction=0.20):
    outer_train = np.asarray(outer_train)
    cut = max(1, int(np.floor((1 - validation_fraction) * len(outer_train))))
    if cut >= len(outer_train):
        cut = len(outer_train) - 1
    return outer_train[:cut], outer_train[cut:]


def encoder_only_state(model_state):
    return {
        name: model_state[name]
        for name in ["thermal_gru", "thermal_projection"]
    }


def fem_encoder_pretraining(
    bank_train, bank_validation, label_version, seed,
    balanced_by_elevation=False, interpolation_lambda=None,
):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    axis_index = AXIS_INDEX["EL"]
    if interpolation_lambda is None:
        key = "Y_v1" if label_version == "v1" else "Y_v2"
        y_train = bank_train[key][:, axis_index].copy()
        y_validation = bank_validation[key][:, axis_index].copy()
    else:
        y_train = (
            bank_train["Y_v2"][:, axis_index]
            + interpolation_lambda * (
                bank_train["Y_v1"][:, axis_index] - bank_train["Y_v2"][:, axis_index]
            )
        )
        y_validation = (
            bank_validation["Y_v2"][:, axis_index]
            + interpolation_lambda * (
                bank_validation["Y_v1"][:, axis_index] - bank_validation["Y_v2"][:, axis_index]
            )
        )

    if balanced_by_elevation:
        train_elevation = bank_train["keys"]["elevation"].to_numpy(dtype=float)
        validation_elevation = bank_validation["keys"]["elevation"].to_numpy(dtype=float)
        train_scaled = np.empty_like(y_train, dtype="float32")
        validation_scaled = np.empty_like(y_validation, dtype="float32")
        for elevation in COMMON_ELEVATIONS:
            train_mask = np.isclose(train_elevation, elevation)
            validation_mask = np.isclose(validation_elevation, elevation)
            mean = float(y_train[train_mask].mean())
            std = max(float(y_train[train_mask].std()), 1e-6)
            train_scaled[train_mask] = (y_train[train_mask] - mean) / std
            validation_scaled[validation_mask] = (y_validation[validation_mask] - mean) / std
    else:
        scaler = StandardScaler().fit(y_train.reshape(-1, 1))
        train_scaled = scaler.transform(y_train.reshape(-1, 1)).reshape(-1)
        validation_scaled = scaler.transform(y_validation.reshape(-1, 1)).reshape(-1)

    model = make_regressor(bank_train["Z"].shape[-1], bank_train["C"].shape[-1])
    model.compile(keras.optimizers.Adam(3e-4, clipnorm=1.0), loss=keras.losses.Huber())
    model.fit(
        [bank_train["Z"], bank_train["C"]], train_scaled,
        validation_data=(
            [bank_validation["Z"], bank_validation["C"]], validation_scaled
        ),
        epochs=PRETRAIN_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks(), verbose=0,
    )
    return encoder_only_state(layer_state(model))


def initialise_real_model(input_dim, context_dim, seed, transfer_state=None):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    model = make_regressor(input_dim, context_dim)
    if transfer_state is not None:
        restore_state(model, transfer_state)
    return model


def best_epoch(history):
    return int(np.argmin(history.history["val_loss"]) + 1)


def select_nested_schedule(
    Z, C, y, inner_train, inner_validation, seed, transfer_state=None,
):
    scaler = StandardScaler().fit(y[inner_train].reshape(-1, 1))
    y_train_scaled = scaler.transform(y[inner_train].reshape(-1, 1))
    y_validation_scaled = scaler.transform(y[inner_validation].reshape(-1, 1))
    model = initialise_real_model(Z.shape[-1], C.shape[-1], seed, transfer_state)
    head_epochs = 0
    if transfer_state is not None:
        model.get_layer("thermal_gru").trainable = False
        model.get_layer("thermal_projection").trainable = False
        model.compile(keras.optimizers.Adam(5e-4, clipnorm=1.0), loss=keras.losses.Huber())
        history = model.fit(
            [Z[inner_train], C[inner_train]], y_train_scaled,
            validation_data=([Z[inner_validation], C[inner_validation]], y_validation_scaled),
            epochs=HEAD_EPOCHS, batch_size=64, callbacks=callbacks(6), verbose=0,
        )
        head_epochs = best_epoch(history)
        model.get_layer("thermal_gru").trainable = True
        model.get_layer("thermal_projection").trainable = True
    model.compile(keras.optimizers.Adam(1e-4, clipnorm=1.0), loss=keras.losses.Huber())
    history = model.fit(
        [Z[inner_train], C[inner_train]], y_train_scaled,
        validation_data=([Z[inner_validation], C[inner_validation]], y_validation_scaled),
        epochs=FINETUNE_EPOCHS, batch_size=64, callbacks=callbacks(10), verbose=0,
    )
    fine_epochs = best_epoch(history)
    validation_prediction = scaler.inverse_transform(
        model.predict([Z[inner_validation], C[inner_validation]], verbose=0)
    ).reshape(-1)
    return head_epochs, fine_epochs, validation_prediction


def refit_fixed_schedule(
    Z, C, y, outer_train, outer_eval, seed, head_epochs, fine_epochs,
    transfer_state=None,
):
    scaler = StandardScaler().fit(y[outer_train].reshape(-1, 1))
    y_scaled = scaler.transform(y[outer_train].reshape(-1, 1))
    model = initialise_real_model(Z.shape[-1], C.shape[-1], seed, transfer_state)
    if transfer_state is not None and head_epochs > 0:
        model.get_layer("thermal_gru").trainable = False
        model.get_layer("thermal_projection").trainable = False
        model.compile(keras.optimizers.Adam(5e-4, clipnorm=1.0), loss=keras.losses.Huber())
        model.fit(
            [Z[outer_train], C[outer_train]], y_scaled,
            epochs=head_epochs, batch_size=64, verbose=0,
        )
        model.get_layer("thermal_gru").trainable = True
        model.get_layer("thermal_projection").trainable = True
    model.compile(keras.optimizers.Adam(1e-4, clipnorm=1.0), loss=keras.losses.Huber())
    model.fit(
        [Z[outer_train], C[outer_train]], y_scaled,
        epochs=fine_epochs, batch_size=64, verbose=0,
    )
    prediction = scaler.inverse_transform(
        model.predict([Z[outer_eval], C[outer_eval]], verbose=0)
    ).reshape(-1)
    return prediction


def nested_prediction(Z, C, y, outer_train, outer_eval, seed, transfer_state=None):
    inner_train, inner_validation = inner_time_split(outer_train)
    head_epochs, fine_epochs, _ = select_nested_schedule(
        Z, C, y, inner_train, inner_validation, seed, transfer_state
    )
    prediction = refit_fixed_schedule(
        Z, C, y, outer_train, outer_eval, seed,
        head_epochs, fine_epochs, transfer_state,
    )
    return prediction, head_epochs, fine_epochs

In [ ]:
# 10. Test A: F1A contro F2A su EL, 24k e CV annidata
dev_idx = np.flatnonzero(~meta["Split"].eq("test"))
nested_rows, schedule_rows = [], []
nested_state_cache, nested_fold_cache = {}, {}

for fold, (outer_train, outer_eval) in enumerate(temporal_fold_bounds(dev_idx), 1):
    fold_projector = RealDomainThermalProjector().fit(X_real[outer_train])
    fold_Z = fold_projector.transform(X_real)
    fold_fem_train = materialize_fem(
        train_candidate_ids[:max(FEM_WINDOW_CURVE)], fold_projector
    )
    fold_fem_validation = materialize_fem(validation_candidate_ids, fold_projector)
    fold_ssl, _ = load_ssl_sequences(
        fold_projector, meta.loc[outer_eval, "Timestamp"].min(),
        RANDOM_SEEDS[0] + fold,
    )
    nested_fold_cache[fold] = {
        "Z": fold_Z, "outer_train": outer_train, "outer_eval": outer_eval,
        "fem_train": fold_fem_train, "fem_validation": fold_fem_validation,
    }

    ridge_prediction = production_prediction("EL", outer_train, outer_eval)
    append_predictions(
        nested_rows, "nested_test_A", "EL", "production_Ridge", -1,
        fold, 0, outer_eval, ridge_prediction,
    )

    for seed in RANDOM_SEEDS:
        ssl_state = run_ssl_pretraining(fold_ssl, seed)
        f1a_state = fem_encoder_pretraining(
            fold_fem_train, fold_fem_validation, "v1", seed
        )
        f2a_state = fem_encoder_pretraining(
            fold_fem_train, fold_fem_validation, "v2", seed
        )
        nested_state_cache[(fold, seed, "real_SSL")] = ssl_state
        nested_state_cache[(fold, seed, "F1A")] = f1a_state
        nested_state_cache[(fold, seed, "F2A")] = f2a_state
        for variant, state in [
            ("real_random_init", None), ("real_SSL", ssl_state),
            ("F1A", f1a_state), ("F2A", f2a_state),
        ]:
            prediction, head_epochs, fine_epochs = nested_prediction(
                fold_Z, C_real, y_real["EL"], outer_train, outer_eval,
                seed, state,
            )
            append_predictions(
                nested_rows, "nested_test_A", "EL", variant, seed,
                fold, 24_000 if variant in ["F1A", "F2A"] else 0,
                outer_eval, prediction,
            )
            schedule_rows.append({
                "fold": fold, "seed": seed, "variant": variant,
                "inner_train_count": len(inner_time_split(outer_train)[0]),
                "inner_validation_count": len(inner_time_split(outer_train)[1]),
                "outer_eval_count": len(outer_eval),
                "head_epochs": head_epochs, "finetune_epochs": fine_epochs,
                "outer_eval_used_for_early_stopping": False,
            })
    gc.collect()

nested_control_predictions_df = pd.DataFrame(nested_rows)
nested_schedule_df = pd.DataFrame(schedule_rows)
nested_control_metrics_df = metrics_from_prediction_rows(nested_control_predictions_df)
display(nested_control_metrics_df.sort_values(["fold", "MAE"]))
display(nested_schedule_df)

In [ ]:
# 11. Riepilogo, bootstrap e gate del Test A
def summarise_oof(predictions, stage):
    subset = predictions.loc[predictions["stage"].eq(stage)].copy()
    metrics = metrics_from_prediction_rows(subset)
    summary = (
        metrics.groupby("variant", as_index=False)
        .agg(
            folds=("fold", "nunique"), seeds=("seed", "nunique"),
            MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
            bias_mean=("bias", "mean"), p90_mean=("p90_abs_error", "mean"),
            within_5_mean=("within_5_percent", "mean"),
        ).sort_values("MAE_mean")
    )
    ensemble = (
        subset.groupby(
            ["variant", "row_index", "Timestamp", "Elevation", "target"],
            as_index=False,
        )["prediction"].mean()
    )
    return metrics, summary, ensemble


nested_control_metrics_df, nested_control_summary_df, nested_ensemble = summarise_oof(
    nested_control_predictions_df, "nested_test_A"
)
bootstrap_rows = []
for candidate, reference in [
    ("F2A", "F1A"), ("F2A", "real_SSL"),
    ("F2A", "real_random_init"), ("F2A", "production_Ridge"),
]:
    wide = nested_ensemble.loc[
        nested_ensemble["variant"].isin([candidate, reference])
    ].pivot_table(
        index=["row_index", "Timestamp", "target"], columns="variant",
        values="prediction", aggfunc="first",
    ).reset_index().dropna(subset=[candidate, reference])
    bootstrap_rows.append({
        "candidate": candidate, "reference": reference,
        **day_block_bootstrap(
            wide["target"], wide[candidate], wide[reference], wide["Timestamp"]
        ),
    })
nested_control_bootstrap_df = pd.DataFrame(bootstrap_rows)

summary_index = nested_control_summary_df.set_index("variant")
f1a_mae = float(summary_index.loc["F1A", "MAE_mean"])
f2a_mae = float(summary_index.loc["F2A", "MAE_mean"])
required_gain = max(MATERIAL_ABSOLUTE_ARCSEC, MATERIAL_RELATIVE * f1a_mae)
paired = nested_control_bootstrap_df.query(
    "candidate == 'F2A' and reference == 'F1A'"
).iloc[0]
fold_wide = nested_control_metrics_df.loc[
    nested_control_metrics_df["variant"].isin(["F1A", "F2A"])
].groupby(["fold", "variant"])["MAE"].mean().unstack()
worst_fold_delta = float((fold_wide["F2A"] - fold_wide["F1A"]).max())
p90_allowance = max(MATERIAL_ABSOLUTE_ARCSEC, MATERIAL_RELATIVE * summary_index.loc["F1A", "p90_mean"])
bias_allowance = max(MATERIAL_ABSOLUTE_ARCSEC, MATERIAL_RELATIVE * abs(summary_index.loc["F1A", "bias_mean"]))
checks = {
    "required_gain_passed": f2a_mae <= f1a_mae - required_gain,
    "paired_ci95_high_below_zero": float(paired["ci95_high"]) < 0,
    "beats_random_init": f2a_mae
    < float(summary_index.loc["real_random_init", "MAE_mean"]),
    "not_worse_than_real_SSL": f2a_mae <= float(summary_index.loc["real_SSL", "MAE_mean"]),
    "no_outer_fold_worse_than_0_25": worst_fold_delta <= MAX_SINGLE_FOLD_REGRESSION,
    "p90_not_materially_worse": float(summary_index.loc["F2A", "p90_mean"]) <= float(summary_index.loc["F1A", "p90_mean"]) + p90_allowance,
    "bias_not_materially_worse": abs(float(summary_index.loc["F2A", "bias_mean"])) <= abs(float(summary_index.loc["F1A", "bias_mean"])) + bias_allowance,
}
nested_control_gate_df = pd.DataFrame([{
    "candidate": "F2A", "reference": "F1A",
    "required_MAE_gain": required_gain, "observed_MAE_gain": f1a_mae - f2a_mae,
    "paired_ci95_low": paired["ci95_low"], "paired_ci95_high": paired["ci95_high"],
    "worst_outer_fold_delta": worst_fold_delta, **checks,
    "all_gates_passed": bool(all(checks.values())),
}])
display(nested_control_summary_df)
display(nested_control_bootstrap_df)
display(nested_control_gate_df)

In [ ]:
# 12. Test B condizionale: target FEM standardizzati per elevazione
balanced_control_predictions_df = pd.DataFrame()
balanced_control_summary_df = pd.DataFrame()
balanced_control_bootstrap_df = pd.DataFrame()
run_balanced = bool(
    RUN_BALANCED_IF_TEST_A_PASSES
    and nested_control_gate_df.iloc[0]["all_gates_passed"]
)
if run_balanced:
    rows = []
    for fold, cache in nested_fold_cache.items():
        for seed in RANDOM_SEEDS:
            for variant, label_version in [("F1A_balanced", "v1"), ("F2A_balanced", "v2")]:
                state = fem_encoder_pretraining(
                    cache["fem_train"], cache["fem_validation"],
                    label_version, seed, balanced_by_elevation=True,
                )
                prediction, _, _ = nested_prediction(
                    cache["Z"], C_real, y_real["EL"],
                    cache["outer_train"], cache["outer_eval"], seed, state,
                )
                append_predictions(
                    rows, "nested_test_B", "EL", variant, seed, fold,
                    24_000, cache["outer_eval"], prediction,
                )
    balanced_control_predictions_df = pd.DataFrame(rows)
    _, balanced_control_summary_df, balanced_ensemble = summarise_oof(
        balanced_control_predictions_df, "nested_test_B"
    )
    wide = balanced_ensemble.pivot_table(
        index=["row_index", "Timestamp", "target"], columns="variant",
        values="prediction", aggfunc="first",
    ).reset_index()
    balanced_control_bootstrap_df = pd.DataFrame([{
        "candidate": "F2A_balanced", "reference": "F1A_balanced",
        **day_block_bootstrap(
            wide["target"], wide["F2A_balanced"],
            wide["F1A_balanced"], wide["Timestamp"],
        ),
    }])
    display(balanced_control_summary_df)
    display(balanced_control_bootstrap_df)
else:
    print("Test B non eseguito: il Test A non ha superato tutti i gate.")

In [ ]:
# 13. Test C: preflight degli operatori fisici sulla telemetria reale
operator_candidates = {
    "v1": [
        ROOT / "calibration" / "physics_02" / "fem_v1_operator_30_60_85.npz",
        ROOT / "calibration" / "fem_v1_operator_30_60_85.npz",
    ],
    "v2": [
        ROOT / "calibration" / "physics_02" / "fem_v2_common_only_operator_30_60_85.npz",
        ROOT / "calibration" / "fem_v2_common_only_operator_30_60_85.npz",
    ],
}
operator_rows = []
selected_operators = {}
for variant, candidates in operator_candidates.items():
    existing = next((path for path in candidates if path.exists()), None)
    selected_operators[variant] = existing
    operator_rows.append({
        "variant": variant, "available": existing is not None,
        "selected_path": str(existing) if existing else None,
        "searched_paths": " | ".join(map(str, candidates)),
    })
real_telemetry_operator_preflight_df = pd.DataFrame(operator_rows)
display(real_telemetry_operator_preflight_df)
if not all(path is not None for path in selected_operators.values()):
    print(
        "Test C bloccato correttamente: servono entrambi gli operatori fisici v1 e v2. "
        "Non viene sostituito alcun operatore con un regressore surrogato."
    )

In [ ]:
# 14. Test D: correttore residuale cross-fitted della Ridge
def crossfitted_ridge_residuals(outer_train):
    rows = []
    for inner_fold, (fit_idx, eval_idx) in enumerate(
        temporal_fold_bounds(np.asarray(outer_train)), 1
    ):
        prediction = production_prediction("EL", fit_idx, eval_idx)
        for row_index, value in zip(eval_idx, prediction):
            rows.append({
                "row_index": int(row_index), "inner_fold": inner_fold,
                "ridge_prediction": float(value),
                "residual_target": float(y_real["EL"][row_index] - value),
            })
    return pd.DataFrame(rows).sort_values("row_index").drop_duplicates("row_index")


residual_corrector_predictions_df = pd.DataFrame()
residual_lambda_df = pd.DataFrame()
if RUN_RESIDUAL_CORRECTOR:
    residual_rows, lambda_rows = [], []
    for fold, cache in nested_fold_cache.items():
        oof = crossfitted_ridge_residuals(cache["outer_train"])
        residual_indices = oof["row_index"].to_numpy(dtype=int)
        residual_target = np.full(len(meta), np.nan, dtype="float32")
        residual_target[residual_indices] = oof["residual_target"].to_numpy(dtype="float32")
        residual_train, residual_validation = inner_time_split(residual_indices)
        ridge_outer_prediction = production_prediction(
            "EL", cache["outer_train"], cache["outer_eval"]
        )
        append_predictions(
            residual_rows, "nested_residual", "EL", "production_Ridge",
            -1, fold, 0, cache["outer_eval"], ridge_outer_prediction,
        )
        for seed in RANDOM_SEEDS:
            for variant, state_key in [
                ("residual_random", None),
                ("residual_F1A", "F1A"),
                ("residual_F2A", "F2A"),
            ]:
                state = None if state_key is None else nested_state_cache[(fold, seed, state_key)]
                head_epochs, fine_epochs, validation_residual_prediction = select_nested_schedule(
                    cache["Z"], C_real, residual_target,
                    residual_train, residual_validation, seed, state,
                )
                validation_target = residual_target[residual_validation]
                lambda_mae = {
                    value: float(np.abs(
                        validation_target - value * validation_residual_prediction
                    ).mean()) for value in LAMBDA_GRID
                }
                selected_lambda = min(lambda_mae, key=lambda value: (lambda_mae[value], value))
                outer_residual_prediction = refit_fixed_schedule(
                    cache["Z"], C_real, residual_target,
                    residual_indices, cache["outer_eval"], seed,
                    head_epochs, fine_epochs, state,
                )
                final_prediction = ridge_outer_prediction + selected_lambda * outer_residual_prediction
                append_predictions(
                    residual_rows, "nested_residual", "EL", variant, seed,
                    fold, 24_000 if state_key else 0,
                    cache["outer_eval"], final_prediction,
                )
                lambda_rows.append({
                    "fold": fold, "seed": seed, "variant": variant,
                    "selected_lambda": selected_lambda,
                    "lambda_zero_available": True,
                    "crossfitted_residual_train_count": len(residual_indices),
                    "outer_eval_used_for_selection": False,
                })
    residual_corrector_predictions_df = pd.DataFrame(residual_rows)
    residual_lambda_df = pd.DataFrame(lambda_rows)

residual_corrector_summary_df = pd.DataFrame()
residual_corrector_bootstrap_df = pd.DataFrame()
residual_corrector_gate_df = pd.DataFrame()
if len(residual_corrector_predictions_df):
    residual_corrector_metrics_df, residual_corrector_summary_df, residual_ensemble = summarise_oof(
        residual_corrector_predictions_df, "nested_residual"
    )
    bootstrap_rows = []
    for candidate in ["residual_random", "residual_F1A", "residual_F2A"]:
        wide = residual_ensemble.loc[
            residual_ensemble["variant"].isin([candidate, "production_Ridge"])
        ].pivot_table(
            index=["row_index", "Timestamp", "target"], columns="variant",
            values="prediction", aggfunc="first",
        ).reset_index()
        bootstrap_rows.append({
            "candidate": candidate, "reference": "production_Ridge",
            **day_block_bootstrap(
                wide["target"], wide[candidate],
                wide["production_Ridge"], wide["Timestamp"],
            ),
        })
    residual_corrector_bootstrap_df = pd.DataFrame(bootstrap_rows)

    summary_index = residual_corrector_summary_df.set_index("variant")
    reference = summary_index.loc["production_Ridge"]
    required_gain = max(
        MATERIAL_ABSOLUTE_ARCSEC,
        MATERIAL_RELATIVE * float(reference["MAE_mean"]),
    )
    p90_allowance = max(
        MATERIAL_ABSOLUTE_ARCSEC,
        MATERIAL_RELATIVE * float(reference["p90_mean"]),
    )
    bias_allowance = max(
        MATERIAL_ABSOLUTE_ARCSEC,
        MATERIAL_RELATIVE * abs(float(reference["bias_mean"])),
    )
    gate_rows = []
    for candidate in ["residual_random", "residual_F1A", "residual_F2A"]:
        candidate_summary = summary_index.loc[candidate]
        paired = residual_corrector_bootstrap_df.loc[
            residual_corrector_bootstrap_df["candidate"].eq(candidate)
        ].iloc[0]
        fold_wide = residual_corrector_metrics_df.loc[
            residual_corrector_metrics_df["variant"].isin(
                [candidate, "production_Ridge"]
            )
        ].groupby(["fold", "variant"])["MAE"].mean().unstack()
        worst_fold_delta = float(
            (fold_wide[candidate] - fold_wide["production_Ridge"]).max()
        )
        checks = {
            "required_gain_passed": float(candidate_summary["MAE_mean"])
            <= float(reference["MAE_mean"]) - required_gain,
            "paired_ci95_high_below_zero": float(paired["ci95_high"]) < 0,
            "no_outer_fold_worse_than_0_25": worst_fold_delta
            <= MAX_SINGLE_FOLD_REGRESSION,
            "p90_not_materially_worse": float(candidate_summary["p90_mean"])
            <= float(reference["p90_mean"]) + p90_allowance,
            "bias_not_materially_worse": abs(float(candidate_summary["bias_mean"]))
            <= abs(float(reference["bias_mean"])) + bias_allowance,
        }
        gate_rows.append({
            "candidate": candidate,
            "reference": "production_Ridge",
            "required_MAE_gain": required_gain,
            "observed_MAE_gain": float(reference["MAE_mean"])
            - float(candidate_summary["MAE_mean"]),
            "paired_ci95_low": paired["ci95_low"],
            "paired_ci95_high": paired["ci95_high"],
            "worst_outer_fold_delta": worst_fold_delta,
            **checks,
            "all_gates_passed": bool(all(checks.values())),
        })
    residual_corrector_gate_df = pd.DataFrame(gate_rows).sort_values(
        ["all_gates_passed", "observed_MAE_gain"], ascending=[False, False]
    )
    display(residual_corrector_summary_df)
    display(residual_corrector_bootstrap_df)
    display(residual_corrector_gate_df)
    display(residual_lambda_df)

In [ ]:
# 15. Test E condizionale: interpolazione fisica Y_lambda
physical_lambda_predictions_df = pd.DataFrame()
physical_lambda_selection_df = pd.DataFrame()
physical_lambda_metrics_df = pd.DataFrame()
physical_lambda_summary_df = pd.DataFrame()
physical_lambda_bootstrap_df = pd.DataFrame()
physical_lambda_gate_df = pd.DataFrame()
run_physical_lambda = bool(
    RUN_PHYSICAL_LAMBDA_IF_TEST_A_PASSES
    and nested_control_gate_df.iloc[0]["all_gates_passed"]
)
if run_physical_lambda:
    lambda_rows, selection_rows = [], []
    for fold, cache in nested_fold_cache.items():
        inner_train, inner_validation = inner_time_split(cache["outer_train"])
        for seed in RANDOM_SEEDS:
            candidates = []
            for interpolation_lambda in LAMBDA_GRID:
                state = fem_encoder_pretraining(
                    cache["fem_train"], cache["fem_validation"],
                    "v2", seed, interpolation_lambda=interpolation_lambda,
                )
                head_epochs, fine_epochs, validation_prediction = select_nested_schedule(
                    cache["Z"], C_real, y_real["EL"],
                    inner_train, inner_validation, seed, state,
                )
                validation_mae = float(np.abs(
                    y_real["EL"][inner_validation] - validation_prediction
                ).mean())
                candidates.append({
                    "lambda": interpolation_lambda,
                    "state": state,
                    "head_epochs": head_epochs,
                    "fine_epochs": fine_epochs,
                    "inner_validation_MAE": validation_mae,
                })
            selected = min(
                candidates,
                key=lambda row: (row["inner_validation_MAE"], row["lambda"]),
            )
            prediction = refit_fixed_schedule(
                cache["Z"], C_real, y_real["EL"],
                cache["outer_train"], cache["outer_eval"], seed,
                selected["head_epochs"], selected["fine_epochs"], selected["state"],
            )
            append_predictions(
                lambda_rows, "nested_physical_lambda", "EL",
                "Y_lambda_selected", seed, fold, 24_000,
                cache["outer_eval"], prediction,
            )
            selection_rows.append({
                "fold": fold, "seed": seed,
                "selected_lambda": selected["lambda"],
                "inner_validation_MAE": selected["inner_validation_MAE"],
                "head_epochs": selected["head_epochs"],
                "finetune_epochs": selected["fine_epochs"],
                "outer_eval_used_for_selection": False,
            })
    physical_lambda_predictions_df = pd.DataFrame(lambda_rows)
    physical_lambda_selection_df = pd.DataFrame(selection_rows)
    f2a_reference = nested_control_predictions_df.loc[
        nested_control_predictions_df["variant"].eq("F2A")
    ].copy()
    f2a_reference["stage"] = "nested_physical_lambda"
    combined = pd.concat(
        [physical_lambda_predictions_df, f2a_reference], ignore_index=True
    )
    physical_lambda_metrics_df, physical_lambda_summary_df, physical_lambda_ensemble = summarise_oof(
        combined, "nested_physical_lambda"
    )
    wide = physical_lambda_ensemble.pivot_table(
        index=["row_index", "Timestamp", "target"], columns="variant",
        values="prediction", aggfunc="first",
    ).reset_index()
    physical_lambda_bootstrap_df = pd.DataFrame([{
        "candidate": "Y_lambda_selected", "reference": "F2A",
        **day_block_bootstrap(
            wide["target"], wide["Y_lambda_selected"],
            wide["F2A"], wide["Timestamp"],
        ),
    }])
    summary_index = physical_lambda_summary_df.set_index("variant")
    candidate_summary = summary_index.loc["Y_lambda_selected"]
    reference_summary = summary_index.loc["F2A"]
    paired = physical_lambda_bootstrap_df.iloc[0]
    required_gain = max(
        MATERIAL_ABSOLUTE_ARCSEC,
        MATERIAL_RELATIVE * float(reference_summary["MAE_mean"]),
    )
    fold_wide = physical_lambda_metrics_df.groupby(
        ["fold", "variant"]
    )["MAE"].mean().unstack()
    worst_fold_delta = float(
        (fold_wide["Y_lambda_selected"] - fold_wide["F2A"]).max()
    )
    p90_allowance = max(
        MATERIAL_ABSOLUTE_ARCSEC,
        MATERIAL_RELATIVE * float(reference_summary["p90_mean"]),
    )
    bias_allowance = max(
        MATERIAL_ABSOLUTE_ARCSEC,
        MATERIAL_RELATIVE * abs(float(reference_summary["bias_mean"])),
    )
    checks = {
        "required_gain_passed": float(candidate_summary["MAE_mean"])
        <= float(reference_summary["MAE_mean"]) - required_gain,
        "paired_ci95_high_below_zero": float(paired["ci95_high"]) < 0,
        "no_outer_fold_worse_than_0_25": worst_fold_delta
        <= MAX_SINGLE_FOLD_REGRESSION,
        "p90_not_materially_worse": float(candidate_summary["p90_mean"])
        <= float(reference_summary["p90_mean"]) + p90_allowance,
        "bias_not_materially_worse": abs(float(candidate_summary["bias_mean"]))
        <= abs(float(reference_summary["bias_mean"])) + bias_allowance,
    }
    physical_lambda_gate_df = pd.DataFrame([{
        "candidate": "Y_lambda_selected", "reference": "F2A",
        "required_MAE_gain": required_gain,
        "observed_MAE_gain": float(reference_summary["MAE_mean"])
        - float(candidate_summary["MAE_mean"]),
        "paired_ci95_low": paired["ci95_low"],
        "paired_ci95_high": paired["ci95_high"],
        "worst_outer_fold_delta": worst_fold_delta,
        **checks,
        "all_gates_passed": bool(all(checks.values())),
    }])
    display(physical_lambda_selection_df)
    display(physical_lambda_summary_df)
    display(physical_lambda_bootstrap_df)
    display(physical_lambda_gate_df)
else:
    print("Test E non eseguito: il Test A non ha superato tutti i gate.")

In [ ]:
# 16. Decisione, criterio di arresto ed export
test_a_passed = bool(nested_control_gate_df.iloc[0]["all_gates_passed"])
ridge_mae = float(
    nested_control_summary_df.set_index("variant").loc["production_Ridge", "MAE_mean"]
)
residual_supported = False
best_residual = None
if len(residual_corrector_gate_df):
    supported = residual_corrector_gate_df.loc[
        residual_corrector_gate_df["all_gates_passed"]
    ]
    residual_supported = bool(len(supported))
    selected_name = (
        supported.iloc[0]["candidate"]
        if residual_supported
        else residual_corrector_gate_df.iloc[0]["candidate"]
    )
    best_residual = residual_corrector_summary_df.loc[
        residual_corrector_summary_df["variant"].eq(selected_name)
    ].iloc[0]

post_training_decision_df = pd.DataFrame([{
    "test_A_F2A_vs_F1A_passed": test_a_passed,
    "test_B_executed": run_balanced,
    "test_C_operator_v1_available": selected_operators["v1"] is not None,
    "test_C_operator_v2_available": selected_operators["v2"] is not None,
    "test_E_physical_lambda_executed": run_physical_lambda,
    "test_E_physical_lambda_supported": bool(
        len(physical_lambda_gate_df)
        and physical_lambda_gate_df.iloc[0]["all_gates_passed"]
    ),
    "best_residual_candidate": None if best_residual is None else best_residual["variant"],
    "best_residual_CV_MAE": np.nan if best_residual is None else best_residual["MAE_mean"],
    "production_Ridge_CV_MAE": ridge_mae,
    "residual_corrector_supported": residual_supported,
    "unlock_locked_test": False,
    "recommendation": (
        "stop_FEM_for_predictive_improvement"
        if not test_a_passed and not residual_supported
        else "continue_only_with_supported_branch"
    ),
}])
display(post_training_decision_df)

OUTPUT_POST = ROOT / "outputs" / "experiments" / "fem_v2_post_training_controls"
OUTPUT_POST.mkdir(parents=True, exist_ok=True)
exports = {
    "nested_control_predictions.csv": nested_control_predictions_df,
    "nested_schedule.csv": nested_schedule_df,
    "nested_control_metrics.csv": nested_control_metrics_df,
    "nested_control_summary.csv": nested_control_summary_df,
    "nested_control_bootstrap.csv": nested_control_bootstrap_df,
    "nested_control_gate.csv": nested_control_gate_df,
    "balanced_control_predictions.csv": balanced_control_predictions_df,
    "balanced_control_summary.csv": balanced_control_summary_df,
    "balanced_control_bootstrap.csv": balanced_control_bootstrap_df,
    "real_telemetry_operator_preflight.csv": real_telemetry_operator_preflight_df,
    "residual_corrector_predictions.csv": residual_corrector_predictions_df,
    "residual_corrector_summary.csv": residual_corrector_summary_df,
    "residual_corrector_bootstrap.csv": residual_corrector_bootstrap_df,
    "residual_corrector_gate.csv": residual_corrector_gate_df,
    "residual_lambda.csv": residual_lambda_df,
    "physical_lambda_predictions.csv": physical_lambda_predictions_df,
    "physical_lambda_selection.csv": physical_lambda_selection_df,
    "physical_lambda_metrics.csv": physical_lambda_metrics_df,
    "physical_lambda_summary.csv": physical_lambda_summary_df,
    "physical_lambda_bootstrap.csv": physical_lambda_bootstrap_df,
    "physical_lambda_gate.csv": physical_lambda_gate_df,
    "post_training_decision.csv": post_training_decision_df,
}
for name, frame in exports.items():
    if frame is not None and len(frame):
        frame.to_csv(OUTPUT_POST / name, index=False)
print("Export completato in:", OUTPUT_POST)

# 17. Output da condividere

Incollare questi output, senza eseguire alcun test bloccato:

1. `nested_schedule_df`
2. `nested_control_metrics_df`
3. `nested_control_summary_df`
4. `nested_control_bootstrap_df`
5. `nested_control_gate_df`
6. `balanced_control_summary_df` e `balanced_control_bootstrap_df`, se eseguiti
7. `real_telemetry_operator_preflight_df`
8. `residual_corrector_summary_df`
9. `residual_corrector_bootstrap_df`
10. `residual_corrector_gate_df`
11. `residual_lambda_df`
12. `physical_lambda_selection_df`, `physical_lambda_summary_df` e
    `physical_lambda_bootstrap_df`, `physical_lambda_gate_df`, se eseguiti
13. `post_training_decision_df`

Il notebook non contiene una cella per sbloccare il test reale.